# Week 4 — 로더 비교 실험 v3 (Loader Experiments)

**목적**: v2에서 확정한 청킹(G2_ko4_en3, 클렌징 없음)을 고정하고, **로더만** PyMuPDFLoader vs pymupdf4llm로 바꿔 검색·답변 품질을 비교한다. ("한 노트북 = 한 변수" 원칙)

| config | 로더 | 청킹 |
|---|---|---|
| L1_pymupdf | PyMuPDFLoader | G2 (한 540/80, 영 620/90) |
| L2_pymupdf4llm | pymupdf4llm (markdown) | G2 (동일) |

**관전 포인트**: 표는 한국어 가이드라인에 집중돼 있으므로(데이터 진단), 표를 markdown으로 보존하는 pymupdf4llm의 효과는 주로 **한국어 문항**에서 나타날 것으로 예상.

> judge는 v2(Sonnet 5)와 달리 **claude-haiku-4-5**를 쓴다(비용 절감). 따라서 v3 점수는 **v3 안에서만(L1 vs L2) 상대 비교**하고, v2 절대값과 직접 비교하지 않는다.

---
## 1. 설정 (CONFIG)

In [1]:
from pathlib import Path
import os, json
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_EVAL = PROJECT_ROOT / "data" / "eval"
VECTOR_ROOT = PROJECT_ROOT / "data" / "vector_store"
for p in [DATA_PROCESSED, DATA_EVAL, VECTOR_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env")
load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY가 .env에 없거나 로드 안 됨"
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY가 .env에 없거나 로드 안 됨"

# ---- 고정값: 생성 gpt-4o-mini / 채점 Haiku 4.5 (v3는 비용 절감 위해 Sonnet에서 하향) ----
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
GEN_MODEL = "gpt-4o-mini"        # 답변 생성: OpenAI (1회)
JUDGE_MODEL = "claude-haiku-4-5" # RAGAS 채점: Anthropic Haiku (1회) — v2 Sonnet5와 다름, v3 내부 비교용
EMBED_DEVICE = "cpu"
TOP_K = 5

# ---- 청킹은 v2에서 확정한 G2로 고정 (로더만 변수로 격리) ----
CHUNK_SIZE_BY_LANG    = {"ko": 540, "en": 620, "unknown": 580}
CHUNK_OVERLAP_BY_LANG = {"ko": 80,  "en": 90,  "unknown": 85}

# ---- 로더 실험 대상 ----
LOADERS = {
    "L1_pymupdf":     "pymupdf",      # baseline 로더 (일반 텍스트)
    "L2_pymupdf4llm": "pymupdf4llm",  # markdown, 표 보존
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("GEN/JUDGE:", GEN_MODEL, "/", JUDGE_MODEL)
print("청킹(고정 G2):", CHUNK_SIZE_BY_LANG)
print("loaders:", list(LOADERS.keys()))

PROJECT_ROOT: /Users/jian/Documents/rag-agent-portfolio
GEN/JUDGE: gpt-4o-mini / claude-haiku-4-5
청킹(고정 G2): {'ko': 540, 'en': 620, 'unknown': 580}
loaders: ['L1_pymupdf', 'L2_pymupdf4llm']


---
## 2. 문서 로딩 (두 로더 각각)

같은 PDF를 두 방식으로 읽어 각각 Document 리스트를 만든다. pymupdf4llm는 manifest 메타를 자동으로 못 붙이므로, 두 로더 모두 filename 기준으로 메타(org/title/language)를 동일하게 부착한다.

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
import re

manifest_path = DATA_RAW / "metadata" / "manifest.json"
if manifest_path.exists():
    with open(manifest_path, encoding="utf-8") as f:
        manifest = json.load(f)
    meta_lookup = {m["filename"]: m for m in manifest if m.get("downloaded")}
else:
    meta_lookup = {}


def _attach_meta(d: Document, pdf_path: Path, page_no):
    extra = meta_lookup.get(pdf_path.name, {})
    d.metadata.update({
        "filename": pdf_path.name,
        "source_folder": pdf_path.parent.name,
        "org": extra.get("org", pdf_path.parent.name),
        "title": extra.get("title", pdf_path.stem),
        "language": extra.get("language", "unknown"),
        "doc_type": extra.get("doc_type", "unknown"),
        "priority": extra.get("priority", "unknown"),
        "page": page_no,
    })
    return d


def load_pymupdf(pdf_root: Path) -> list:
    docs = []
    for pdf_path in pdf_root.rglob("*.pdf"):
        for d in PyMuPDFLoader(str(pdf_path)).load():
            _attach_meta(d, pdf_path, d.metadata.get("page", "?"))
            docs.append(d)
    return docs


def load_pymupdf4llm(pdf_root: Path) -> list:
    import pymupdf4llm
    docs = []
    for pdf_path in pdf_root.rglob("*.pdf"):
        # page_chunks=True -> 페이지 단위 dict 리스트 (text + metadata)
        pages = pymupdf4llm.to_markdown(str(pdf_path), page_chunks=True, show_progress=False)
        for i, pg in enumerate(pages):
            text = pg.get("text", "") if isinstance(pg, dict) else str(pg)
            pmeta = pg.get("metadata", {}) if isinstance(pg, dict) else {}
            d = Document(page_content=text, metadata={})
            _attach_meta(d, pdf_path, pmeta.get("page", i + 1))
            docs.append(d)
    return docs


def guess_lang(text: str) -> str:
    kr = len(re.findall(r"[\uac00-\ud7a3]", text))
    return "ko" if kr > 20 else "en"


LOADER_FUNCS = {"pymupdf": load_pymupdf, "pymupdf4llm": load_pymupdf4llm}

doc_store = {}
for name, kind in LOADERS.items():
    docs = LOADER_FUNCS[kind](DATA_RAW / "pdf")
    # language 메타 fallback (두 로더 동일 처리)
    for d in docs:
        if d.metadata.get("language") in (None, "unknown", "?"):
            d.metadata["language"] = guess_lang(d.page_content)
    doc_store[name] = docs
    avg = sum(len(d.page_content) for d in docs) / max(len(docs), 1)
    print(f"{name:16s} 페이지 Document={len(docs):4d}  평균길이(문자)={avg:.0f}")

L1_pymupdf       페이지 Document= 796  평균길이(문자)=1532
Consider using the pymupdf_layout package for a greatly improved page layout analysis.
L2_pymupdf4llm   페이지 Document= 796  평균길이(문자)=1440


---
## 3. 청킹 (G2 고정, 두 로더 동일 분할기)

로더만 변수로 격리하기 위해 **분할기는 L1·L2 동일**하게 쓴다. markdown의 O(n²) 청킹 폭주를 막기 위해 separator에서 `""`(빈 문자열)을 빼고, markdown 헤더(`## `, `### `)를 분할 지점으로 추가한다.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# "" 제거(markdown 청킹 폭주 방지) + markdown 헤더 분할점 추가. L1/L2 동일 적용.
SEPARATORS = ["\n## ", "\n### ", "\n\n", "\n", ". ", " "]


def make_splitter(size: int, overlap: int) -> RecursiveCharacterTextSplitter:
    return RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap,
        separators=SEPARATORS, length_function=len,
    )


def split_by_lang(docs) -> list:
    out = []
    for lang in set(d.metadata.get("language", "unknown") for d in docs):
        size = CHUNK_SIZE_BY_LANG.get(lang, CHUNK_SIZE_BY_LANG["unknown"])
        overlap = CHUNK_OVERLAP_BY_LANG.get(lang, CHUNK_OVERLAP_BY_LANG["unknown"])
        sub = [d for d in docs if d.metadata.get("language", "unknown") == lang]
        out.extend(make_splitter(size, overlap).split_documents(sub))
    return out


chunk_store = {}
for name in LOADERS:
    ck = split_by_lang(doc_store[name])
    chunk_store[name] = ck
    avg = sum(len(c.page_content) for c in ck) / max(len(ck), 1)
    print(f"{name:16s} chunk수={len(ck):5d}  평균길이(문자)={avg:.0f}")

L1_pymupdf       chunk수= 2773  평균길이(문자)=477
L2_pymupdf4llm   chunk수= 2795  평균길이(문자)=430


---
## 4. 임베딩 & 리트리버 (컬렉션 week4v3_*)

v2 인덱스(`week4v2_*`)와 충돌하지 않도록 `week4v3_*` 컬렉션을 새로 만든다. 로더가 달라 chunk 내용이 다르므로 반드시 새로 인덱싱된다.

In [4]:
import os
from huggingface_hub import snapshot_download

os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)
os.environ["HF_HUB_ETAG_TIMEOUT"] = "10"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

try:
    model_dir = snapshot_download(repo_id=EMBEDDING_MODEL, local_files_only=True)
    print("로컬 캐시 snapshot:", model_dir)
except Exception as e:
    print("로컬 캐시 불완전 -> 다운로드:", repr(e))
    model_dir = snapshot_download(repo_id=EMBEDDING_MODEL, local_files_only=False, max_workers=1)

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import chromadb

embeddings = HuggingFaceEmbeddings(
    model_name=model_dir,
    model_kwargs={"device": EMBED_DEVICE},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16},
)
print("embedding 로드 완료")


def build_retriever(name: str):
    vdir = VECTOR_ROOT / f"week4v3_{name}"
    vdir.mkdir(parents=True, exist_ok=True)
    coll = f"breast_rag_week4v3_{name}"
    client = chromadb.PersistentClient(path=str(vdir))
    if coll in [c.name for c in client.list_collections()]:
        vs = Chroma(collection_name=coll, embedding_function=embeddings, persist_directory=str(vdir))
        print(f"  {name}: 기존 컬렉션 재사용 ({vs._collection.count()}개)")
    else:
        ck = chunk_store[name]
        print(f"  {name}: 신규 인덱싱 {len(ck)}개 ... (CPU면 시간 소요)")
        vs = Chroma.from_documents(documents=ck, embedding=embeddings,
                                   collection_name=coll, persist_directory=str(vdir))
    return vs.as_retriever(search_kwargs={"k": TOP_K})

로컬 캐시 snapshot: /Users/jian/.cache/huggingface/hub/models--intfloat--multilingual-e5-base/snapshots/d128750597153bb5987e10b1c3493a34e5a4502a
embedding 로드 완료


---
## 5. 답변 생성 (gpt-4o-mini, v2와 동일 프롬프트)

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=GEN_MODEL, temperature=0)

RAG_PROMPT = ChatPromptTemplate.from_template(
    """당신은 유방암 정보 검색 보조 시스템입니다.

아래 [참고 문서]만 사용해서 [질문]에 답변하세요. 문서에 없는 내용은 추측하지 말고 "제공된 문서에서 확인할 수 없습니다"라고 답하세요.
답변 마지막에는 반드시 다음 두 가지를 포함하세요:
1. 출처: 참고한 문서명과 페이지 (예: 출처: 국립암센터 유방암 검진 권고안, p.5)
2. 면책 문구: "이 답변은 일반 정보 제공 목적이며, 실제 진단·치료는 반드시 의료진과 상의하세요."

[참고 문서]
{context}

[질문]
{question}

[답변]"""
)


def format_context(docs):
    parts = []
    for i, d in enumerate(docs, 1):
        m = d.metadata
        parts.append(f"[{i}] 출처: {m.get('org','?')} / {m.get('title','?')} / p.{m.get('page','?')}\n{d.page_content}")
    return "\n\n---\n\n".join(parts)


def make_ask(retriever):
    def ask(question: str):
        docs = retriever.invoke(question)
        prompt = RAG_PROMPT.format(context=format_context(docs), question=question)
        answer = (llm | StrOutputParser()).invoke(prompt)
        return {"question": question, "answer": answer, "contexts": [d.page_content for d in docs]}
    return ask

---
## 6. 평가셋 로딩 (golden_set_v1, 30문항)

In [6]:
import pandas as pd

golden_path = DATA_EVAL / "golden_set_v1.csv"
if golden_path.exists():
    df_golden = pd.read_csv(golden_path)
    print(f"golden_set_v1 로드: {len(df_golden)}문항")
else:
    raise FileNotFoundError("golden_set_v1.csv 없음 — data/eval/ 에 넣으세요.")
golden = df_golden.to_dict("records")
df_golden[["question"]]

golden_set_v1 로드: 30문항


,question
0,유방암 검진은 몇 살부터 받는 것이 권장되나요?
1,유방암의 주요 위험 요인은 무엇인가요?
2,HER2 양성 유방암이란 무엇인가요?
3,유방암 1기와 2기의 차이는 무엇인가요?
4,DCIS(유관상피내암)는 침윤성 유방암과 어떻게 다른가요?
5,항호르몬제 치료는 어떤 환자에게 사용되나요?
6,유방절제술 후 재건수술에는 어떤 방법이 있나요?
7,BRCA 유전자 검사는 누구에게 권장되나요?
8,전이성 유방암인 4기의 일반적인 치료 목표는 무엇인가요?
9,유방암 환자가 식이요법에서 주의할 점은 무엇인가요?


---
## 7. 로더별 실행 + RAGAS 평가 (judge=Haiku)

In [7]:
import nest_asyncio
nest_asyncio.apply()  # Jupyter async 충돌로 인한 RAGAS NaN 방지

from ragas import evaluate, EvaluationDataset
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from tqdm import tqdm
from langchain_anthropic import ChatAnthropic

# Sonnet5 때 필요했던 temperature 차단 래퍼 — Haiku엔 불필요하지만 무해하므로 그대로 사용
class ChatAnthropicNoTemp(ChatAnthropic):
    def __setattr__(self, name, value):
        if name == "temperature":
            value = None
        super().__setattr__(name, value)

ragas_llm = LangchainLLMWrapper(ChatAnthropicNoTemp(model=JUDGE_MODEL, max_tokens=4096))
ragas_emb = LangchainEmbeddingsWrapper(embeddings)
METRICS = [faithfulness, answer_relevancy, context_precision]
METRIC_COLS = ["faithfulness", "answer_relevancy", "context_precision"]


def run_loader(name: str) -> pd.DataFrame:
    retriever = build_retriever(name)
    ask = make_ask(retriever)
    rows = []
    for g in tqdm(golden, desc=f"RAG[{name}]"):
        r = ask(g["question"])
        rows.append({"question": r["question"], "answer": r["answer"],
                     "contexts": r["contexts"], "ground_truth": g.get("ground_truth", "")})
    samples = [{"user_input": r["question"], "response": r["answer"],
                "retrieved_contexts": r["contexts"], "reference": r["ground_truth"]} for r in rows]
    ds = EvaluationDataset.from_list(samples)
    scores = evaluate(dataset=ds, metrics=METRICS, llm=ragas_llm, embeddings=ragas_emb)
    df = scores.to_pandas()
    df.to_csv(DATA_PROCESSED / f"week4_ragas_{name}_v3.csv", index=False, encoding="utf-8-sig")
    return df


score_tables = {}
for name in LOADERS:
    score_tables[name] = run_loader(name)
    print(f"{name}: 완료")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


  L1_pymupdf: 신규 인덱싱 2773개 ... (CPU면 시간 소요)


RAG[L1_pymupdf]:   0%|          | 0/30 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
RAG[L1_pymupdf]: 100%|██████████| 30/30 [01:32<00:00,  3.07s/it]


Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


L1_pymupdf: 완료
  L2_pymupdf4llm: 신규 인덱싱 2795개 ... (CPU면 시간 소요)


RAG[L2_pymupdf4llm]: 100%|██████████| 30/30 [01:25<00:00,  2.86s/it]


Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]

L2_pymupdf4llm: 완료


---
## 8. 비교 (L1 vs L2, 언어별)

로더만 다르므로 두 config의 차이가 곧 **로더 효과**다. 표가 한국어에 몰려 있으니 pymupdf4llm의 이득은 한국어 CP에서 보일 것으로 예상.

In [8]:
# 집계 비교표
rows = []
for name in LOADERS:
    df = score_tables[name]
    ck = chunk_store[name]
    row = {"로더": name, "chunk수": len(ck),
           "평균길이": round(sum(len(c.page_content) for c in ck) / max(len(ck), 1))}
    for col in METRIC_COLS:
        row[col] = round(df[col].mean(), 4) if col in df.columns else None
    rows.append(row)
df_compare = pd.DataFrame(rows)

base = df_compare.iloc[0]  # L1을 델타 기준
for col in METRIC_COLS:
    df_compare[col + "_delta"] = (df_compare[col] - base[col]).round(4)

df_compare.to_csv(DATA_PROCESSED / "week4_loader_comparison_v3.csv", index=False, encoding="utf-8-sig")
print("저장: week4_loader_comparison_v3.csv")
df_compare

저장: week4_loader_comparison_v3.csv


,로더,chunk수,평균길이,faithfulness,answer_relevancy,context_precision,faithfulness_delta,answer_relevancy_delta,context_precision_delta
0,L1_pymupdf,2773,477,0.6866,0.8388,0.6955,0.000,0.000,0.0000
1,L2_pymupdf4llm,2795,430,0.6456,0.8068,0.5056,-0.041,-0.032,-0.1899


In [9]:
# 언어별 context_precision 비교 (표 효과는 주로 한국어에서)
import re
def _lang(q): return "KO" if re.search("[가-힣]", str(q)) else "EN"

print("=== 로더별 언어별 Context Precision ===")
for L in ["KO", "EN"]:
    print(f"[{L}]")
    for name in LOADERS:
        df = score_tables[name].copy()
        df["_lang"] = df["user_input"].apply(_lang)
        sub = df[df["_lang"] == L]
        print(f"   {name:16s} CP={sub['context_precision'].mean():.4f}  (n={len(sub)})")
    print()

=== 로더별 언어별 Context Precision ===
[KO]
   L1_pymupdf       CP=0.7577  (n=20)
   L2_pymupdf4llm   CP=0.5215  (n=20)

[EN]
   L1_pymupdf       CP=0.5711  (n=10)
   L2_pymupdf4llm   CP=0.4739  (n=10)

